# Text preprocessing and classification

Compare six text-cleaning variants with Word2Vec, spaCy document vectors, and TF-IDF features, followed by simple classifiers. The experiment requires `combined_data.csv` with `text` and `label` columns; set `SPAM_DATA_PATH` if the CSV is stored elsewhere. The file is not included here. Install the imported libraries and the `en_core_web_sm` spaCy model. NLTK resources are downloaded on first run.

The saved results were generated on the complete dataset after Word2Vec and TF-IDF were changed to fit on training texts only. The small spaCy pipeline does not contain pretrained static word vectors. Preprocessing and models are selected on a validation split; the selected configuration is then refit and evaluated once on a separate test split.

Dataset: [Spam Email Classification Dataset](https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset).

**Data:** [Email Spam Classification Dataset](https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset). Place `combined_data.csv` here or set `SPAM_DATA_PATH`.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(os.environ.get('SPAM_DATA_PATH', 'combined_data.csv'), encoding='utf-8')

df.head()

,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [2]:
import os
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
STEMMER = PorterStemmer()

df = pd.read_csv(os.environ.get('SPAM_DATA_PATH', 'combined_data.csv'), encoding='utf-8')

df = df.drop_duplicates(subset='text')
df = df.dropna()


def clean_basic(text):
    """Только нижний регистр и удаление лишних пробелов"""
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_no_stopwords(text):
    """Без стоп-слов, но с пунктуацией"""
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    stop_words = STOP_WORDS
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

def clean_full(text):
    """Полная очистка: без пунктуации, чисел, стоп-слов"""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    stop_words = STOP_WORDS
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

def clean_with_lemmatization(text):
    """Полная очистка + лемматизация"""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    stop_words = STOP_WORDS
    lemmatizer = LEMMATIZER

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

def clean_with_stemming(text):
    """Полная очистка + стемминг"""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    stop_words = STOP_WORDS
    stemmer = STEMMER

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

def clean_keep_special_chars(text):
    """Сохраняем специальные символы ($, !, ?), которые могут быть важны для спама"""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s\$\!\?\*\%\.]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


In [3]:
df['text_basic'] = df['text'].apply(clean_basic)
df['text_no_stopwords'] = df['text'].apply(clean_no_stopwords)
df['text_full'] = df['text'].apply(clean_full)
df['text_lemmatized'] = df['text'].apply(clean_with_lemmatization)
df['text_stemmed'] = df['text'].apply(clean_with_stemming)
df['text_special_chars'] = df['text'].apply(clean_keep_special_chars)

sample_idx = 10
print("Оригинал:")
print(df['text'].iloc[sample_idx])
print("\n" + "="*50 + "\n")

print("Только нижний регистр:")
print(df['text_basic'].iloc[sample_idx])
print("\n" + "-"*30 + "\n")

print("Без стоп-слов:")
print(df['text_no_stopwords'].iloc[sample_idx])
print("\n" + "-"*30 + "\n")

print("Полная очистка:")
print(df['text_full'].iloc[sample_idx])
print("\n" + "-"*30 + "\n")

print("С лемматизацией:")
print(df['text_lemmatized'].iloc[sample_idx])
print("\n" + "-"*30 + "\n")

print("Со стеммингом:")
print(df['text_stemmed'].iloc[sample_idx])
print("\n" + "-"*30 + "\n")

print("Со спецсимволами:")
print(df['text_special_chars'].iloc[sample_idx])

Оригинал:
my dear fellow do you feel insecure about your penis size you need our new improved megadik penis enlargement pills please read on did you know megadik was featured in leading mens magazines such as fhm maxim plus many others and rated no escapenumber choice for penis enlargement » gain escapenumber inches in length » increase your penis width girth by upto escapenumber » produce stronger rock hard erections » escapenumber safe to take with no side effects » doctor approved and recommended » fast shipping worldwide you have nothing to lose just a lot to gain http slasy net regards escapelong remains in escapelong use escapenumberf milliescapenumberns escapenumberf peescapenumberple in this wescapenumberrld i am escapelong man and alescapenumberng with all thescapenumberse milliescapenumberns although far from perfect especially in that it precludes a vast waldron


Только нижний регистр:
my dear fellow do you feel insecure about your penis size you need our new improved megad

In [4]:
df.head()

,label,text,text_basic,text_no_stopwords,text_full,text_lemmatized,text_stemmed,text_special_chars
0,1,ounce feather bowl hummingbird opec moment ala...,ounce feather bowl hummingbird opec moment ala...,ounce feather bowl hummingbird opec moment ala...,ounce feather bowl hummingbird opec moment ala...,ounce feather bowl hummingbird opec moment ala...,ounc feather bowl hummingbird opec moment alab...,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...,wulvob get your medircations online qnb ikud v...,wulvob get medircations online qnb ikud viagra...,wulvob get medircations online qnb ikud viagra...,wulvob get medircations online qnb ikud viagra...,wulvob get medirc onlin qnb ikud viagra escape...,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...,computer connection from cnn com wednesday esc...,computer connection cnn com wednesday escapenu...,computer connection cnn com wednesday escapenu...,computer connection cnn com wednesday escapenu...,comput connect cnn com wednesday escapenumb ma...,computer connection from cnn com wednesday esc...
3,1,university degree obtain a prosperous future m...,university degree obtain a prosperous future m...,university degree obtain prosperous future mon...,university degree obtain prosperous future mon...,university degree obtain prosperous future mon...,univers degre obtain prosper futur money earn ...,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...,thanks for all your answers guys i know i shou...,thanks answers guys know checked rsync manual ...,thanks answers guys know checked rsync manual ...,thanks answer guy know checked rsync manual wo...,thank answer guy know check rsync manual would...,thanks for all your answers guys i know i shou...


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import word_tokenize
import gensim
from gensim.models import Word2Vec
import spacy


nltk.download('punkt')
nltk.download('punkt_tab')

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

preprocess_cols = [
    'text_basic',
    'text_no_stopwords',
    'text_full',
    'text_lemmatized',
    'text_stemmed',
    'text_special_chars'
]

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM (linear)': LinearSVC(max_iter=2000, dual=False, random_state=42, C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42)
}

embeddings = ['Word2Vec', 'spaCy', 'TF-IDF']

all_results = {}

# Reserve the test set before comparing preprocessing and feature choices.
train_idx, holdout_idx = train_test_split(df.index.to_numpy(), test_size=0.30, random_state=42, stratify=df['label'])
val_idx, test_idx = train_test_split(holdout_idx, test_size=0.50, random_state=42, stratify=df.loc[holdout_idx, 'label'])
print(f'Train: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)}')


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Train: 58412, validation: 12517, test: 12517


In [6]:
def make_features(kind, training_texts, evaluation_texts):
    """Fit data-dependent representations only on the supplied training texts."""
    if kind == "TF-IDF":
        vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
        return vectorizer.fit_transform(training_texts), vectorizer.transform(evaluation_texts)
    if kind == "spaCy":
        def vectors(texts):
            return np.array([doc.vector if len(doc.vector) else np.zeros(96)
                             for doc in nlp.pipe(texts, batch_size=128)])
        return vectors(training_texts), vectors(evaluation_texts)
    train_tokens = [word_tokenize(text) for text in training_texts]
    eval_tokens = [word_tokenize(text) for text in evaluation_texts]
    w2v = Word2Vec(sentences=train_tokens, vector_size=150, window=5,
                   min_count=2, workers=1, epochs=10, seed=42)
    def mean_vector(tokens):
        vectors = [w2v.wv[token] for token in tokens if token in w2v.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(150)
    return (np.array([mean_vector(t) for t in train_tokens]),
            np.array([mean_vector(t) for t in eval_tokens]))

def make_classifier(name, kind):
    if name == "LogisticRegression":
        return LogisticRegression(max_iter=1000, solver="saga" if kind == "TF-IDF" else "lbfgs", random_state=42)
    if name == "SVM (linear)":
        return LinearSVC(max_iter=2000, dual=False, random_state=42)
    if name == "RandomForest" and kind != "TF-IDF":
        return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
    raise ValueError(f"Unsupported model and representation: {name}, {kind}")

def run_experiment(col_name, df):
    print(f"Preprocessing: {col_name}")
    train_text, val_text = df.loc[train_idx, col_name], df.loc[val_idx, col_name]
    y_train, y_val = df.loc[train_idx, "label"], df.loc[val_idx, "label"]
    results = {}
    for kind in ["Word2Vec", "spaCy", "TF-IDF"]:
        X_train, X_val = make_features(kind, train_text, val_text)
        names = ["LogisticRegression", "SVM (linear)"]
        if kind != "TF-IDF": names.append("RandomForest")
        results[kind] = {}
        for name in names:
            estimator = make_classifier(name, kind)
            estimator.fit(X_train, y_train)
            score = accuracy_score(y_val, estimator.predict(X_val))
            results[kind][name] = score
            print(f"{kind} / {name} validation accuracy: {score:.4f}")
    return results

In [7]:
for col in preprocess_cols:
    all_results[col] = run_experiment(col, df)

Preprocessing: text_basic


Word2Vec / LogisticRegression validation accuracy: 0.9698


Word2Vec / SVM (linear) validation accuracy: 0.9676


Word2Vec / RandomForest validation accuracy: 0.9815


spaCy / LogisticRegression validation accuracy: 0.8523


spaCy / SVM (linear) validation accuracy: 0.8502


... (39 outputs left out)

spaCy / SVM (linear) validation accuracy: 0.8569


spaCy / RandomForest validation accuracy: 0.9297


TF-IDF / LogisticRegression validation accuracy: 0.9829


TF-IDF / SVM (linear) validation accuracy: 0.9887


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

candidates = [(score, column, kind, name)
              for column, representations in all_results.items()
              for kind, models in representations.items()
              for name, score in models.items()]
val_score, best_column, best_kind, best_name = max(candidates)
print(f"Selected on validation: {best_column} / {best_kind} / {best_name}: {val_score:.4f}")
# Refit the chosen representation and classifier on train + validation only.
fit_idx = np.concatenate([train_idx, val_idx])
X_fit, X_test = make_features(best_kind, df.loc[fit_idx, best_column], df.loc[test_idx, best_column])
final_model = make_classifier(best_name, best_kind)
final_model.fit(X_fit, df.loc[fit_idx, 'label'])
test_predictions = final_model.predict(X_test)
print(f"Held-out test accuracy: {accuracy_score(df.loc[test_idx, 'label'], test_predictions):.4f}")
print(classification_report(df.loc[test_idx, 'label'], test_predictions, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(df.loc[test_idx, 'label'], test_predictions))

Selected on validation: text_special_chars / TF-IDF / SVM (linear): 0.9887


Held-out test accuracy: 0.9895
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5930
           1       0.99      0.99      0.99      6587

    accuracy                           0.99     12517
   macro avg       0.99      0.99      0.99     12517
weighted avg       0.99      0.99      0.99     12517

Confusion matrix:
[[5847   83]
 [  48 6539]]
